# 2. Feature Engineering — MaintiFlow
Makine kimliği, zaman damgası, türetilmiş özellikler ve arıza tipi etiketi.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

pd.set_option('display.max_columns', None)
df = pd.read_csv('../data/ai4i2020.csv')
df.shape

(10000, 14)

## 2.1 machine_id ve timestamp üretimi
20 makine varsayıyoruz. Veri setindeki satır sırasını (UDI) zaman akışı olarak kabul edip, satırları makinelere sırayla (round-robin) dağıtıyoruz. Her makine kendi içinde 5 dakikalık aralıklarla ölçüm almış gibi zaman damgası alıyor.

In [2]:
N_MACHINES = 20
df = df.sort_values('UDI').reset_index(drop=True)
df['machine_id'] = (df['UDI'] - 1) % N_MACHINES + 1

df['reading_seq'] = df.groupby('machine_id').cumcount()
base_time = datetime(2026, 1, 1, 8, 0, 0)
df['timestamp'] = df['reading_seq'].apply(lambda s: base_time + timedelta(minutes=5 * s))

df[['UDI', 'machine_id', 'reading_seq', 'timestamp']].head(10)

,UDI,machine_id,reading_seq,timestamp
0,1,1,0,2026-01-01 08:00:00
1,2,2,0,2026-01-01 08:00:00
2,3,3,0,2026-01-01 08:00:00
3,4,4,0,2026-01-01 08:00:00
4,5,5,0,2026-01-01 08:00:00
5,6,6,0,2026-01-01 08:00:00
6,7,7,0,2026-01-01 08:00:00
7,8,8,0,2026-01-01 08:00:00
8,9,9,0,2026-01-01 08:00:00
9,10,10,0,2026-01-01 08:00:00


In [3]:
print('Makine basina ortalama satir sayisi:', df.groupby('machine_id').size().mean())
print('Makine basina satir sayisi dagilimi:')
print(df.groupby('machine_id').size())

Makine basina ortalama satir sayisi: 500.0
Makine basina satir sayisi dagilimi:
machine_id
1     500
2     500
3     500
4     500
5     500
6     500
7     500
8     500
9     500
10    500
11    500
12    500
13    500
14    500
15    500
16    500
17    500
18    500
19    500
20    500
dtype: int64


## 2.2 Türetilmiş sensör özellikleri
- **temp_diff**: Process temperature − Air temperature (soğutma sistemi verimliliği ile ilgili, HDF ile bağlantılı)
- **power_w**: Tork × açısal hız (Watt cinsinden), PWF (güç kaynaklı arıza) ile doğrudan ilişkili

In [4]:
df['temp_diff'] = df['Process temperature [K]'] - df['Air temperature [K]']
df['power_w'] = df['Torque [Nm]'] * (df['Rotational speed [rpm]'] * 2 * np.pi / 60)

df[['Air temperature [K]', 'Process temperature [K]', 'temp_diff', 'Torque [Nm]', 'Rotational speed [rpm]', 'power_w']].describe()

,Air temperature [K],Process temperature [K],temp_diff,Torque [Nm],Rotational speed [rpm],power_w
count,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000
mean,300.004930,310.005560,10.000630,39.986910,1538.776100,6279.744953
std,2.000259,1.483734,1.001094,9.968934,179.284096,1067.418295
min,295.300000,305.700000,7.600000,3.800000,1168.000000,1148.440610
25%,298.300000,308.800000,9.300000,33.200000,1423.000000,5561.184484
50%,300.100000,310.100000,9.800000,40.100000,1503.000000,6271.027344
75%,301.500000,311.100000,11.000000,46.800000,1612.000000,7003.002724
max,304.500000,313.800000,12.100000,76.600000,2886.000000,10469.923005


## 2.2b Tool Wear Failure (TWF) için kritik aşınma bandı özelliği

03_model_training'de TWF sınıfı (46 satır) `class_weight='balanced'` ile bile hiç öğrenilemedi. AI4I 2020 veri setinin dokümante edilmiş üretim kuralına göre TWF, takım aşınması ~200 dakikayı geçtiğinde rastgele bir olasılıkla tetikleniyor. Veride bunu doğruluyoruz: aşınma ≥200 olan 801 satırın sadece %5,6'sı (45/46 TWF) gerçekten arızalı — yani asıl sinyal ham `Tool wear` değerinde değil, bu bandın **içinde olup olmama** durumunda.

Ağaç tabanlı bir modelin bu bandı (aşınma ≥200) izole etmesi tek bir ham sayısal eşik değil, sınırlı TWF örneği (train'de ~37 satır) her bootstrap ağacında farklı şekilde bölünmeye çalışıyor. Bunu tek bir binary özellik olarak açıkça vererek modele kısayol sağlıyoruz: `tool_wear_critical`.

In [5]:
df['tool_wear_critical'] = (df['Tool wear [min]'] >= 200).astype(int)

band = df[df['tool_wear_critical'] == 1]
print('Kritik banddaki satir sayisi:', len(band))
print('Kritik bandda TWF orani:', round(band['TWF'].mean(), 4))
print('Kritik bandda yakalanan TWF sayisi (toplam 46 icinden):', band['TWF'].sum())

Kritik banddaki satir sayisi: 801
Kritik bandda TWF orani: 0.0562
Kritik bandda yakalanan TWF sayisi (toplam 46 icinden): 45


## 2.3 Arıza tipi etiketi (failure_type)
TWF/HDF/PWF/OSF arasından ilk pozitif olanı seçiyoruz (ARIZA_PARCA tablosunda RNF olmadığı için RNF'i ayrı bir bayrakla işaretliyoruz, failure_type'a dahil etmiyoruz).

In [6]:
type_cols = ['TWF', 'HDF', 'PWF', 'OSF']

def pick_failure_type(row):
    for c in type_cols:
        if row[c] == 1:
            return c
    return 'NoFailure'

df['failure_type'] = df.apply(pick_failure_type, axis=1)
df['is_rnf_only'] = (df['RNF'] == 1) & (df[type_cols].sum(axis=1) == 0)

print(df['failure_type'].value_counts())
print('\nSadece RNF kaynakli, tipi belirlenemeyen ariza sayisi:', df['is_rnf_only'].sum())
print('\nBu satirlarda Machine failure degeri:')
print(df.loc[df['is_rnf_only'], 'Machine failure'].value_counts())

failure_type
NoFailure    9670
HDF           115
PWF            91
OSF            78
TWF            46
Name: count, dtype: int64

Sadece RNF kaynakli, tipi belirlenemeyen ariza sayisi: 18

Bu satirlarda Machine failure degeri:
Machine failure
0    18
Name: count, dtype: int64


## 2.4 Yeni özelliklerin arıza ile ilişkisini kontrol etme
Türettiğimiz özelliklerin gerçekten sinyal taşıyıp taşımadığını görmek için arızalı/sağlam gruplar arasında ortalama karşılaştırması yapıyoruz.

In [7]:
compare_cols = ['temp_diff', 'power_w', 'Tool wear [min]']
print(df.groupby('Machine failure')[compare_cols].mean())

                 temp_diff      power_w  Tool wear [min]
Machine failure                                         
0                10.021571  6244.547534       106.693717
1                 9.403835  7282.819485       143.781711


## 2.5 İşlenmiş veriyi kaydetme

In [8]:
out_cols = ['UDI', 'machine_id', 'timestamp', 'Product ID', 'Type',
            'Air temperature [K]', 'Process temperature [K]', 'temp_diff',
            'Rotational speed [rpm]', 'Torque [Nm]', 'power_w', 'Tool wear [min]',
            'tool_wear_critical', 'Machine failure', 'failure_type', 'is_rnf_only']
processed = df[out_cols]
processed.to_csv('../data/processed.csv', index=False)
print('Kaydedildi:', processed.shape)
processed.head()

Kaydedildi: (10000, 16)


,UDI,machine_id,timestamp,Product ID,Type,Air temperature [K],Process temperature [K],temp_diff,Rotational speed [rpm],Torque [Nm],power_w,Tool wear [min],tool_wear_critical,Machine failure,failure_type,is_rnf_only
0,1,1,2026-01-01 08:00:00,M14860,M,298.1,308.6,10.5,1551,42.8,6951.590560,0,0,0,NoFailure,False
1,2,2,2026-01-01 08:00:00,L47181,L,298.2,308.7,10.5,1408,46.3,6826.722724,3,0,0,NoFailure,False
2,3,3,2026-01-01 08:00:00,L47182,L,298.1,308.5,10.4,1498,49.4,7749.387543,5,0,0,NoFailure,False
3,4,4,2026-01-01 08:00:00,L47183,L,298.2,308.6,10.4,1433,39.5,5927.504659,7,0,0,NoFailure,False
4,5,5,2026-01-01 08:00:00,L47184,L,298.2,308.7,10.5,1408,40.0,5897.816608,9,0,0,NoFailure,False
